Подготовка

In [1]:
!pip install psycopg2-binary sqlalchemy boto3 s3fs pyarrow scikit-learn

INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/

In [2]:
import pandas as pd
from sqlalchemy import create_engine

pg_user = "user"
pg_password = "password"
pg_host = "postgres"
pg_port = "5432"
pg_db = "oil_db"

engine = create_engine(
    f"postgresql+psycopg2://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_db}"
)

tables = pd.read_sql("""
SELECT table_name 
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name
""", engine)

tables

,table_name
0,deliveries
1,drivers
2,oil_stations
3,production
4,pump_failures
5,pump_sensors
6,pumps
7,vehicles
8,well_targets
9,well_telemetry


In [3]:
wells = pd.read_sql("SELECT * FROM wells", engine)
production = pd.read_sql("SELECT * FROM production", engine)
telemetry = pd.read_sql("SELECT * FROM well_telemetry", engine)

print("wells:", wells.shape)
print("production:", production.shape)
print("telemetry:", telemetry.shape)

display(wells.head())
display(production.head())
display(telemetry.head())

wells: (5, 7)
production: (150, 10)
telemetry: (48, 10)


,well_id,name,field_name,region,start_date,operator,status
0,1,Well-101,Severo-Ural,Khanty-Mansi,2018-03-15,UralOil,active
1,2,Well-102,Severo-Ural,Khanty-Mansi,2019-07-22,UralOil,active
2,3,Well-203,Vostok-Field,Yamalo-Nenets,2020-02-11,GazPromTech,maintenance
3,4,Well-304,Zapad-Field,Tomsk,2017-05-01,RosDrill,suspended
4,5,Well-305,Zapad-Field,Tomsk,2019-11-03,RosDrill,active


,prod_id,well_id,date,oil_ton,gas_m3,water_m3,energy_kwh,downtime_hours,temperature,pressure
0,1,1,2025-10-01,212.4,55200.0,182.3,7450.0,0.5,88.1,120.4
1,2,1,2025-10-02,213.8,55320.0,181.9,7490.0,0.3,87.8,121.0
2,3,1,2025-10-03,211.9,55040.0,183.2,7430.0,0.7,88.5,119.8
3,4,1,2025-10-04,215.1,55500.0,180.5,7515.0,0.2,87.6,121.5
4,5,1,2025-10-05,214.6,55380.0,182.0,7480.0,0.4,88.0,120.9


,record_id,well_id,timestamp,pump_speed_rpm,pump_current,pressure_in,pressure_out,temperature,vibration,oil_flow_rate
0,1,1,2025-10-01 00:00:00,1470.0,58.2,95.3,122.4,88.1,1.4,8.8
1,2,1,2025-10-01 01:00:00,1468.0,58.5,95.1,122.1,88.2,1.5,8.9
2,3,1,2025-10-01 02:00:00,1472.0,58.0,94.9,121.8,88.3,1.6,8.7
3,4,1,2025-10-01 03:00:00,1475.0,58.3,95.0,122.2,88.4,1.4,8.8
4,5,1,2025-10-01 04:00:00,1473.0,58.1,94.8,121.9,88.2,1.5,8.7


In [4]:
merged_df = production.merge(
    wells,
    on="well_id",
    how="left"
)

print(merged_df.shape)

merged_df.head()

(150, 16)


,prod_id,well_id,date,oil_ton,gas_m3,water_m3,energy_kwh,downtime_hours,temperature,pressure,name,field_name,region,start_date,operator,status
0,1,1,2025-10-01,212.4,55200.0,182.3,7450.0,0.5,88.1,120.4,Well-101,Severo-Ural,Khanty-Mansi,2018-03-15,UralOil,active
1,2,1,2025-10-02,213.8,55320.0,181.9,7490.0,0.3,87.8,121.0,Well-101,Severo-Ural,Khanty-Mansi,2018-03-15,UralOil,active
2,3,1,2025-10-03,211.9,55040.0,183.2,7430.0,0.7,88.5,119.8,Well-101,Severo-Ural,Khanty-Mansi,2018-03-15,UralOil,active
3,4,1,2025-10-04,215.1,55500.0,180.5,7515.0,0.2,87.6,121.5,Well-101,Severo-Ural,Khanty-Mansi,2018-03-15,UralOil,active
4,5,1,2025-10-05,214.6,55380.0,182.0,7480.0,0.4,88.0,120.9,Well-101,Severo-Ural,Khanty-Mansi,2018-03-15,UralOil,active


In [5]:
merged_df.isnull().sum()

prod_id            0
well_id            0
date               0
oil_ton            0
gas_m3             0
water_m3           0
energy_kwh         0
downtime_hours     0
temperature       30
pressure          30
name               0
field_name         0
region             0
start_date         0
operator           0
status             0
dtype: int64

In [6]:
merged_df = merged_df.fillna(0)


In [7]:
merged_df = merged_df[
    (merged_df["temperature"] < 200) &
    (merged_df["pressure"] < 300)
]

In [8]:
merged_df["downtime_ratio"] = (
    merged_df["downtime_hours"] / 24
)

merged_df["avg_pressure"] = merged_df["pressure"]

merged_df["avg_temperature"] = merged_df["temperature"]

merged_df.head()

,prod_id,well_id,date,oil_ton,gas_m3,water_m3,energy_kwh,downtime_hours,temperature,pressure,name,field_name,region,start_date,operator,status,downtime_ratio,avg_pressure,avg_temperature
0,1,1,2025-10-01,212.4,55200.0,182.3,7450.0,0.5,88.1,120.4,Well-101,Severo-Ural,Khanty-Mansi,2018-03-15,UralOil,active,0.020833,120.4,88.1
1,2,1,2025-10-02,213.8,55320.0,181.9,7490.0,0.3,87.8,121.0,Well-101,Severo-Ural,Khanty-Mansi,2018-03-15,UralOil,active,0.012500,121.0,87.8
2,3,1,2025-10-03,211.9,55040.0,183.2,7430.0,0.7,88.5,119.8,Well-101,Severo-Ural,Khanty-Mansi,2018-03-15,UralOil,active,0.029167,119.8,88.5
3,4,1,2025-10-04,215.1,55500.0,180.5,7515.0,0.2,87.6,121.5,Well-101,Severo-Ural,Khanty-Mansi,2018-03-15,UralOil,active,0.008333,121.5,87.6
4,5,1,2025-10-05,214.6,55380.0,182.0,7480.0,0.4,88.0,120.9,Well-101,Severo-Ural,Khanty-Mansi,2018-03-15,UralOil,active,0.016667,120.9,88.0


In [9]:
well_analytics = merged_df.groupby("well_id").agg({
    "oil_ton": "mean",
    "gas_m3": "mean",
    "water_m3": "mean",
    "avg_pressure": "mean",
    "avg_temperature": "mean",
    "downtime_ratio": "mean"
}).reset_index()

well_analytics.head()

,well_id,oil_ton,gas_m3,water_m3,avg_pressure,avg_temperature,downtime_ratio
0,1,213.150000,55218.333333,182.430000,120.543333,88.206667,0.020139
1,2,185.816667,49772.666667,161.356667,115.306667,84.486667,0.030278
2,3,121.696667,40150.000000,121.083333,107.160000,79.433333,0.081111
3,4,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
4,5,198.430000,52258.333333,165.976667,119.426667,86.800000,0.018194


In [10]:
print(well_analytics.shape)
well_analytics.describe()

(5, 7)


,well_id,oil_ton,gas_m3,water_m3,avg_pressure,avg_temperature,downtime_ratio
count,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,3.000000,143.818667,39479.866667,126.169333,92.487333,67.785333,0.229944
std,1.581139,87.644531,22781.786727,74.048300,51.968218,38.039618,0.431237
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.018194
25%,2.000000,121.696667,40150.000000,121.083333,107.160000,79.433333,0.020139
50%,3.000000,185.816667,49772.666667,161.356667,115.306667,84.486667,0.030278
75%,4.000000,198.430000,52258.333333,165.976667,119.426667,86.800000,0.081111
max,5.000000,213.150000,55218.333333,182.430000,120.543333,88.206667,1.000000


In [11]:
import boto3
from io import BytesIO

In [12]:
s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minio",
    aws_secret_access_key="minio123"
)

print("MinIO connected")


MinIO connected


In [13]:
bucket_name = "oil-data"

existing_buckets = [
    bucket["Name"]
    for bucket in s3.list_buckets()["Buckets"]
]

if bucket_name not in existing_buckets:
    s3.create_bucket(Bucket=bucket_name)

print("Bucket ready")

Bucket ready


In [14]:
well_analytics["partition_id"] = (
    "well_" + well_analytics["well_id"].astype(str)
)

well_analytics.head()

,well_id,oil_ton,gas_m3,water_m3,avg_pressure,avg_temperature,downtime_ratio,partition_id
0,1,213.150000,55218.333333,182.430000,120.543333,88.206667,0.020139,well_1
1,2,185.816667,49772.666667,161.356667,115.306667,84.486667,0.030278,well_2
2,3,121.696667,40150.000000,121.083333,107.160000,79.433333,0.081111,well_3
3,4,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,well_4
4,5,198.430000,52258.333333,165.976667,119.426667,86.800000,0.018194,well_5


In [15]:
for partition in well_analytics["partition_id"].unique():

    partition_df = well_analytics[
        well_analytics["partition_id"] == partition
    ]

    parquet_buffer = BytesIO()

    partition_df.to_parquet(
        parquet_buffer,
        index=False
    )

    s3.put_object(
        Bucket=bucket_name,
        Key=f"{partition}/analytics.parquet",
        Body=parquet_buffer.getvalue()
    )

print("Parquet uploaded to MinIO")

Parquet uploaded to MinIO


In [16]:
objects = s3.list_objects(
    Bucket=bucket_name
)

for obj in objects["Contents"]:
    print(obj["Key"])

well_1/analytics.parquet
well_2/analytics.parquet
well_3/analytics.parquet
well_4/analytics.parquet
well_5/analytics.parquet


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

features = [
    "gas_m3",
    "water_m3",
    "avg_pressure",
    "avg_temperature",
    "downtime_ratio"
]

target = "oil_ton"

X = well_analytics[features]
y = well_analytics[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("R2:", r2)

MAE: 5.734964823267106
R2: 0.1304456451495224


In [18]:
well_analytics.to_sql(
    "well_analytics_mart",
    engine,
    if_exists="replace",
    index=False
)

print("Data mart created")

Data mart created


In [19]:
mart_df = pd.read_sql(
    "SELECT * FROM well_analytics_mart",
    engine
)

mart_df.head()


,well_id,oil_ton,gas_m3,water_m3,avg_pressure,avg_temperature,downtime_ratio,partition_id
0,1,213.150000,55218.333333,182.430000,120.543333,88.206667,0.020139,well_1
1,2,185.816667,49772.666667,161.356667,115.306667,84.486667,0.030278,well_2
2,3,121.696667,40150.000000,121.083333,107.160000,79.433333,0.081111,well_3
3,4,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,well_4
4,5,198.430000,52258.333333,165.976667,119.426667,86.800000,0.018194,well_5


Задание 1 — Аналитика добычи

In [20]:
# DAILY PRODUCTION MART

production["date"] = pd.to_datetime(
    production["date"]
)

daily_production_mart = production.groupby("date").agg(
    total_oil_ton=("oil_ton", "sum"),
    total_gas_m3=("gas_m3", "sum"),
    total_water_m3=("water_m3", "sum"),
    avg_pressure=("pressure", "mean"),
    avg_temperature=("temperature", "mean"),
    total_energy_kwh=("energy_kwh", "sum"),
    total_downtime_hours=("downtime_hours", "sum")
).reset_index()

daily_production_mart.head()

,date,total_oil_ton,total_gas_m3,total_water_m3,avg_pressure,avg_temperature,total_energy_kwh,total_downtime_hours
0,2025-10-01,717.5,197170.0,631.1,115.425,84.625,26730.0,27.7
1,2025-10-02,717.3,197200.0,630.5,115.475,84.625,26735.0,28.0
2,2025-10-03,719.2,197350.0,630.8,115.500,84.875,26770.0,27.8
3,2025-10-04,722.9,197810.0,627.4,116.000,84.375,26875.0,26.7
4,2025-10-05,721.0,197620.0,630.1,115.725,84.675,26810.0,27.5


In [21]:
# WELL KPI MART

well_kpi_mart = production.groupby("well_id").agg(
    avg_oil_ton=("oil_ton", "mean"),
    avg_gas_m3=("gas_m3", "mean"),
    avg_water_m3=("water_m3", "mean"),
    avg_pressure=("pressure", "mean"),
    avg_temperature=("temperature", "mean"),
    avg_energy_kwh=("energy_kwh", "mean"),
    avg_downtime_hours=("downtime_hours", "mean")
).reset_index()

# процент простоя
well_kpi_mart["downtime_percent"] = (
    well_kpi_mart["avg_downtime_hours"] / 24 * 100
)

well_kpi_mart.head()

,well_id,avg_oil_ton,avg_gas_m3,avg_water_m3,avg_pressure,avg_temperature,avg_energy_kwh,avg_downtime_hours,downtime_percent
0,1,213.150000,55218.333333,182.430000,120.543333,88.206667,7458.833333,0.483333,2.013889
1,2,185.816667,49772.666667,161.356667,115.306667,84.486667,6794.166667,0.726667,3.027778
2,3,121.696667,40150.000000,121.083333,107.160000,79.433333,5285.166667,1.946667,8.111111
3,4,0.000000,0.000000,0.000000,NaN,NaN,0.000000,24.000000,100.000000
4,5,198.430000,52258.333333,165.976667,119.426667,86.800000,7240.166667,0.436667,1.819444


In [22]:
# TOP WELLS

top_wells = well_kpi_mart.sort_values(
    by="avg_oil_ton",
    ascending=False
)

top_wells.head()

,well_id,avg_oil_ton,avg_gas_m3,avg_water_m3,avg_pressure,avg_temperature,avg_energy_kwh,avg_downtime_hours,downtime_percent
0,1,213.150000,55218.333333,182.430000,120.543333,88.206667,7458.833333,0.483333,2.013889
4,5,198.430000,52258.333333,165.976667,119.426667,86.800000,7240.166667,0.436667,1.819444
1,2,185.816667,49772.666667,161.356667,115.306667,84.486667,6794.166667,0.726667,3.027778
2,3,121.696667,40150.000000,121.083333,107.160000,79.433333,5285.166667,1.946667,8.111111
3,4,0.000000,0.000000,0.000000,NaN,NaN,0.000000,24.000000,100.000000


In [23]:
# WORST WELLS

worst_wells = well_kpi_mart.sort_values(
    by="avg_oil_ton",
    ascending=True
)

worst_wells.head()

,well_id,avg_oil_ton,avg_gas_m3,avg_water_m3,avg_pressure,avg_temperature,avg_energy_kwh,avg_downtime_hours,downtime_percent
3,4,0.000000,0.000000,0.000000,NaN,NaN,0.000000,24.000000,100.000000
2,3,121.696667,40150.000000,121.083333,107.160000,79.433333,5285.166667,1.946667,8.111111
1,2,185.816667,49772.666667,161.356667,115.306667,84.486667,6794.166667,0.726667,3.027778
4,5,198.430000,52258.333333,165.976667,119.426667,86.800000,7240.166667,0.436667,1.819444
0,1,213.150000,55218.333333,182.430000,120.543333,88.206667,7458.833333,0.483333,2.013889


In [24]:
daily_production_mart.to_sql(
    "daily_production_mart",
    engine,
    if_exists="replace",
    index=False
)

well_kpi_mart.to_sql(
    "well_kpi_mart",
    engine,
    if_exists="replace",
    index=False
)

top_wells.to_sql(
    "top_wells_mart",
    engine,
    if_exists="replace",
    index=False
)

worst_wells.to_sql(
    "worst_wells_mart",
    engine,
    if_exists="replace",
    index=False
)

print("All marts loaded")

All marts loaded


Задание 2 — Прогноз дебита

In [25]:
well_targets = pd.read_sql(
    "SELECT * FROM well_targets",
    engine
)

print(well_targets.shape)

well_targets.head()

(90, 3)


,well_id,date,daily_oil_ton
0,1,2025-10-01,212.4
1,1,2025-10-02,213.8
2,1,2025-10-03,211.9
3,1,2025-10-04,215.1
4,1,2025-10-05,214.6


In [26]:
telemetry["timestamp"] = pd.to_datetime(
    telemetry["timestamp"]
)

telemetry["date"] = telemetry["timestamp"].dt.date

telemetry_daily = telemetry.groupby(
    ["well_id", "date"]
).agg(
    avg_pressure_out=("pressure_out", "mean"),
    avg_temperature=("temperature", "mean"),
    avg_pump_current=("pump_current", "mean"),
    avg_pump_speed=("pump_speed_rpm", "mean"),
    avg_oil_flow_rate=("oil_flow_rate", "mean"),
    avg_vibration=("vibration", "mean")
).reset_index()

telemetry_daily.head()

,well_id,date,avg_pressure_out,avg_temperature,avg_pump_current,avg_pump_speed,avg_oil_flow_rate,avg_vibration
0,1,2025-10-01,122.233333,88.2875,58.395833,1472.541667,8.808333,1.462500
1,2,2025-10-01,115.516667,84.3500,54.650000,1430.666667,7.537500,1.441667


In [27]:
well_targets["date"] = pd.to_datetime(
    well_targets["date"]
).dt.date

In [28]:
ml_dataset = telemetry_daily.merge(
    well_targets,
    on=["well_id", "date"],
    how="inner"
)

print(ml_dataset.shape)

ml_dataset.head()


(2, 9)


,well_id,date,avg_pressure_out,avg_temperature,avg_pump_current,avg_pump_speed,avg_oil_flow_rate,avg_vibration,daily_oil_ton
0,1,2025-10-01,122.233333,88.2875,58.395833,1472.541667,8.808333,1.462500,212.4
1,2,2025-10-01,115.516667,84.3500,54.650000,1430.666667,7.537500,1.441667,185.9


In [29]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

features = [
    "avg_pressure_out",
    "avg_temperature",
    "avg_pump_current",
    "avg_pump_speed",
    "avg_oil_flow_rate",
    "avg_vibration"
]

target = "daily_oil_ton"

X = ml_dataset[features]
y = ml_dataset[target]

model = LinearRegression()

model.fit(X, y)

predictions = model.predict(X)

ml_dataset["predicted_oil_ton"] = predictions

mae = mean_absolute_error(y, predictions)

rmse = mean_squared_error(
    y,
    predictions,
    squared=False
)

print("MAE:", mae)
print("RMSE:", rmse)

ml_dataset.head()

MAE: 5.684341886080802e-14
RMSE: 6.355287432313019e-14


,well_id,date,avg_pressure_out,avg_temperature,avg_pump_current,avg_pump_speed,avg_oil_flow_rate,avg_vibration,daily_oil_ton,predicted_oil_ton
0,1,2025-10-01,122.233333,88.2875,58.395833,1472.541667,8.808333,1.462500,212.4,212.4
1,2,2025-10-01,115.516667,84.3500,54.650000,1430.666667,7.537500,1.441667,185.9,185.9


Метрики практически равны нулю - идеальный случай. Связано с тем, что в dataset всего 2 строки и модель обучалась и предсказывали на тех же данных

In [30]:
ml_dataset.to_sql(
    "ml_oil_prediction_mart",
    engine,
    if_exists="replace",
    index=False
)

print("ML mart saved")

ML mart saved


Задание 3

In [31]:
pump_sensors = pd.read_sql(
    "SELECT * FROM pump_sensors",
    engine
)

pump_failures = pd.read_sql(
    "SELECT * FROM pump_failures",
    engine
)

print(pump_sensors.shape)
print(pump_failures.shape)

display(pump_sensors.head())
display(pump_failures.head())

(72, 8)
(3, 5)


,record_id,pump_id,timestamp,temperature,vibration,current,rpm,pressure
0,1,1,2025-10-01 00:00:00,72.3,2.1,58.2,1470.0,122.4
1,2,1,2025-10-01 03:00:00,72.6,2.0,58.4,1472.0,122.5
2,3,1,2025-10-01 06:00:00,73.1,2.2,58.6,1474.0,122.6
3,4,1,2025-10-01 09:00:00,72.8,2.1,58.3,1471.0,122.3
4,5,1,2025-10-01 12:00:00,73.0,2.3,58.5,1473.0,122.7


,failure_id,pump_id,failure_date,failure_type,downtime_hours
0,1,1,2025-10-04 02:00:00,Overheating,6.5
1,2,3,2025-10-04 06:00:00,Bearing vibration,8.0
2,3,5,2025-10-04 08:00:00,Electrical fault,10.2


In [32]:
from scipy.stats import zscore

anomaly_df = pump_sensors.copy()

anomaly_df["temp_zscore"] = zscore(
    anomaly_df["temperature"]
)

anomaly_df["vibration_zscore"] = zscore(
    anomaly_df["vibration"]
)

anomaly_df["current_zscore"] = zscore(
    anomaly_df["current"]
)

# anomaly flag
anomaly_df["is_anomaly"] = (
    (anomaly_df["temp_zscore"].abs() > 2)
    |
    (anomaly_df["vibration_zscore"].abs() > 2)
    |
    (anomaly_df["current_zscore"].abs() > 2)
)

anomaly_df.head()

,record_id,pump_id,timestamp,temperature,vibration,current,rpm,pressure,temp_zscore,vibration_zscore,current_zscore,is_anomaly
0,1,1,2025-10-01 00:00:00,72.3,2.1,58.2,1470.0,122.4,-0.632358,-0.732037,-0.205851,False
1,2,1,2025-10-01 03:00:00,72.6,2.0,58.4,1472.0,122.5,-0.572530,-0.755725,-0.141410,False
2,3,1,2025-10-01 06:00:00,73.1,2.2,58.6,1474.0,122.6,-0.472815,-0.708349,-0.076970,False
3,4,1,2025-10-01 09:00:00,72.8,2.1,58.3,1471.0,122.3,-0.532644,-0.732037,-0.173631,False
4,5,1,2025-10-01 12:00:00,73.0,2.3,58.5,1473.0,122.7,-0.492758,-0.684660,-0.109190,False


In [33]:
anomalies = anomaly_df[
    anomaly_df["is_anomaly"] == True
]

print(anomalies.shape)

anomalies.head()

(6, 12)


,record_id,pump_id,timestamp,temperature,vibration,current,rpm,pressure,temp_zscore,vibration_zscore,current_zscore,is_anomaly
23,24,1,2025-10-03 21:00:00,85.1,9.1,65.2,1508.0,128.3,1.920342,0.926150,2.049557,True
67,68,5,2025-10-03 09:00:00,84.3,14.3,63.8,1522.0,125.5,1.760798,2.157946,1.598475,True
68,69,5,2025-10-03 12:00:00,85.5,15.8,64.2,1525.0,125.8,2.000114,2.513272,1.727356,True
69,70,5,2025-10-03 15:00:00,86.8,17.3,64.6,1528.0,126.0,2.259373,2.868598,1.856236,True
70,71,5,2025-10-03 18:00:00,88.0,18.9,65.1,1531.0,126.3,2.498688,3.247612,2.017337,True


In [34]:
from sklearn.preprocessing import MinMaxScaler

risk_df = anomaly_df.copy()

scaler = MinMaxScaler()

risk_df[[
    "temp_scaled",
    "vibration_scaled",
    "current_scaled"
]] = scaler.fit_transform(
    risk_df[[
        "temperature",
        "vibration",
        "current"
    ]]
)

risk_df["risk_score"] = (
    risk_df["temp_scaled"] * 0.3
    +
    risk_df["vibration_scaled"] * 0.5
    +
    risk_df["current_scaled"] * 0.2
)

risk_df = risk_df.sort_values(
    by="risk_score",
    ascending=False
)

risk_df.head()

,record_id,pump_id,timestamp,temperature,vibration,current,rpm,pressure,temp_zscore,vibration_zscore,current_zscore,is_anomaly,temp_scaled,vibration_scaled,current_scaled,risk_score
71,72,5,2025-10-03 21:00:00,89.3,20.5,65.5,1534.0,126.6,2.757947,3.626626,2.146217,True,1.000000,1.000000,1.000000,1.000000
70,71,5,2025-10-03 18:00:00,88.0,18.9,65.1,1531.0,126.3,2.498688,3.247612,2.017337,True,0.935644,0.914894,0.964912,0.931122
69,70,5,2025-10-03 15:00:00,86.8,17.3,64.6,1528.0,126.0,2.259373,2.868598,1.856236,True,0.876238,0.829787,0.921053,0.861975
68,69,5,2025-10-03 12:00:00,85.5,15.8,64.2,1525.0,125.8,2.000114,2.513272,1.727356,True,0.811881,0.750000,0.885965,0.795757
67,68,5,2025-10-03 09:00:00,84.3,14.3,63.8,1522.0,125.5,1.760798,2.157946,1.598475,True,0.752475,0.670213,0.850877,0.731024


Самый рискованный насос №5, у него одновременно высокие температура, вибрация, ток, максимальный risk_score

In [35]:
risk_df.to_sql(
    "pump_risk_mart",
    engine,
    if_exists="replace",
    index=False
)

print("pump_risk_mart saved")

pump_risk_mart saved


Задание 4

In [37]:
deliveries = pd.read_sql(
    "SELECT * FROM deliveries",
    engine
)

print(deliveries.shape)

deliveries.head()

(30, 12)


,delivery_id,date,source,destination,product_type,volume_ton,cost_usd,delay_hours,distance_km,weather_conditions,driver_id,vehicle_id
0,1,2025-10-01,Base-Khanty,Station-01,Diesel,32.5,2100.50,0.0,180.0,Clear,1,1
1,2,2025-10-01,Base-Tomsk,Station-02,Gasoline,28.0,1850.00,1.5,150.0,Rain,2,2
2,3,2025-10-01,Base-Tyumen,Station-03,Diesel,22.0,1650.25,0.0,120.0,Clear,3,3
3,4,2025-10-02,Base-Khanty,Station-04,Kerosene,35.0,2400.80,2.0,210.0,Fog,4,1
4,5,2025-10-02,Base-Tomsk,Station-05,Gasoline,20.5,1500.00,0.5,110.0,Cloudy,5,2


Cost per km

In [38]:
deliveries["cost_per_km"] = (
    deliveries["cost_usd"]
    /
    deliveries["distance_km"]
)

deliveries.head()

,delivery_id,date,source,destination,product_type,volume_ton,cost_usd,delay_hours,distance_km,weather_conditions,driver_id,vehicle_id,cost_per_km
0,1,2025-10-01,Base-Khanty,Station-01,Diesel,32.5,2100.50,0.0,180.0,Clear,1,1,11.669444
1,2,2025-10-01,Base-Tomsk,Station-02,Gasoline,28.0,1850.00,1.5,150.0,Rain,2,2,12.333333
2,3,2025-10-01,Base-Tyumen,Station-03,Diesel,22.0,1650.25,0.0,120.0,Clear,3,3,13.752083
3,4,2025-10-02,Base-Khanty,Station-04,Kerosene,35.0,2400.80,2.0,210.0,Fog,4,1,11.432381
4,5,2025-10-02,Base-Tomsk,Station-05,Gasoline,20.5,1500.00,0.5,110.0,Cloudy,5,2,13.636364


KPI по водителям

In [39]:
driver_kpi_mart = deliveries.groupby(
    "driver_id"
).agg(
    total_deliveries=("delivery_id", "count"),
    avg_delay_hours=("delay_hours", "mean"),
    avg_cost_per_km=("cost_per_km", "mean"),
    total_volume_ton=("volume_ton", "sum")
).reset_index()

driver_kpi_mart.head()

,driver_id,total_deliveries,avg_delay_hours,avg_cost_per_km,total_volume_ton
0,1,5,0.200000,11.726857,168.4
1,2,7,0.785714,12.299379,195.2
2,3,7,0.428571,13.261150,165.1
3,4,4,0.625000,11.754716,130.7
4,5,7,2.285714,13.587001,141.1


Из таблицы выше делаем вывод, что задержки увеличивают стоимость

Weather Delay Analytics

In [40]:
weather_delay_mart = deliveries.groupby(
    "weather_conditions"
).agg(
    avg_delay_hours=("delay_hours", "mean"),
    avg_cost_usd=("cost_usd", "mean"),
    total_deliveries=("delivery_id", "count")
).reset_index()

weather_delay_mart.head()

,weather_conditions,avg_delay_hours,avg_cost_usd,total_deliveries
0,Clear,0.100,1921.660000,15
1,Cloudy,0.375,1707.650000,4
2,Fog,1.500,2033.600000,3
3,Rain,2.000,1750.200000,5
4,Snow,3.500,1433.466667,3


In [41]:
driver_kpi_mart.to_sql(
    "driver_kpi_mart",
    engine,
    if_exists="replace",
    index=False
)

weather_delay_mart.to_sql(
    "weather_delay_mart",
    engine,
    if_exists="replace",
    index=False
)

deliveries.to_sql(
    "deliveries_mart",
    engine,
    if_exists="replace",
    index=False
)

print("Logistics marts saved")

Logistics marts saved
